# Brain · use 1 · healthy structure segmentation*A small model that names the structures of a healthy brain on T1 MRI, and is honestabout what it gets wrong.*> **For learning anatomy, never for diagnosis.** This notebook trains on labelled public> MRI from [Mindboggle-101](DATA.md) and shows its output beside the dataset's own> labels. It makes no clinical claim, and nothing here may be used to say what a person> has.> **This is a demonstrator.** It trains a very small model, on a few subjects, on a few> 2D slices, in about a minute on a laptop CPU. It is meant to be read end to end in one> sitting. The last section says exactly which numbers to change for a full run, and what> that would cost.**What you will see, in order**1. the healthy anatomy this use is about;2. the data, and where each label comes from;3. a small segmentation model;4. an evaluation that splits by subject;5. the errors, looked at rather than hidden.

## 1. The healthy anatomyA T1-weighted MRI is the sequence anatomy is usually taught on: it separates the tissuesof the brain well. This use asks a model to find, on that image, the structures a studentlearns to recognise first:| Structure | Where it is ||---|---|| **Cerebral cortex** | the folded grey matter at the surface of the hemispheres || **Cerebral white matter** | beneath the cortex, the fibre tissue that connects it || **Lateral ventricles** | the paired CSF spaces at the centre of the hemispheres || **Thalamus** | the deep grey nucleus on each side of the third ventricle || **Hippocampus** | in the medial temporal lobe, along the floor of the temporal horn |The healthy brain comes first in this atlas because every lesion is read against it: tosee that something has displaced the ventricle, you must first know the ventricle.

## 2. The data, and where each label comes fromThe dataset is Mindboggle-101: 101 healthy adults, each with a T1 image and two labelvolumes. [`DATA.md`](DATA.md) records how it is fetched, its checksums, and its terms.The data is **not** in this repository; it sits under `data/`, which git ignores.**One thing matters more than any other here.** The two label volumes do not have thesame provenance:- `labels.DKT31.manual.nii.gz` — the **cortex**, labelled by hand following the DKT  protocol. These are expert labels.- `labels.DKT31.manual+aseg.nii.gz` — the same cortical labels, **plus** the non-cortical  structures taken from FreeSurfer's automatic segmentation (*aseg*). These are a  program's output, not an expert's.So a model trained on the cortex learns from people, and a model trained on theventricles, thalamus or hippocampus learns to **imitate FreeSurfer**, including whereverFreeSurfer is wrong.**A second record fixes part of that.** For 20 of the 101 subjects — the OASIS-TRT-20cohort — the subcortical structures were also labelled by hand, and published separatelyas [doi:10.5281/zenodo.22071825](https://doi.org/10.5281/zenodo.22071825) under a morerestrictive licence (CC BY-NC-ND 4.0). The model of record for this use is trained andmeasured with those labels, which is what lets its score mean *correct* rather than*agrees with a program*. One structure is still beyond reach: **cerebral white matter hasno manual labels anywhere in this dataset**, so it stays automatic, and says so.This notebook keeps the two provenances apart, marks every structure with its own, andrepeats it next to the results.

In [ ]:
from pathlib import Path

# --- what the demonstrator runs on. Raise these for a real run (see the last section).
CONFIG = dict(
    subjects_per_cohort_train=3,   # 5 cohorts -> 15 training subjects
    subjects_per_cohort_test=2,    #           -> 10 held-out subjects
    slices_per_subject=24,         # axial slices, sampled through the labelled range
    image_size=96,                 # 192 mm field of view, downsampled by 2
    epochs=25,
    batch_size=8,
    learning_rate=3e-3,
    base_channels=8,               # width of the small U-Net
    seed=0,
)

DATA = Path("data/extracted/Mindboggle101_release3/Mindboggle101_volumes")
assert DATA.is_dir(), f"data not found at {DATA} - see DATA.md for how to fetch it"
print("data root:", DATA)

In [ ]:
import numpy as np, torch

np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.use_deterministic_algorithms(True, warn_only=True)
print("numpy", np.__version__, "| torch", torch.__version__, "| threads", torch.get_num_threads())

### 2.1 The subjects, and the splitThe split is **by subject**, never by slice. Two slices of the same brain look almostalike; if one went to training and the other to the test set, the score would measurememory, not generalisation. Subjects are also drawn from every cohort, so that trainingand test both span the five acquisition sources.

In [ ]:
cohorts = {d.name.replace("_volumes", ""): sorted(p for p in d.iterdir() if p.is_dir())
           for d in sorted(DATA.iterdir()) if d.is_dir()}

train_subjects, test_subjects = [], []
for name, subs in cohorts.items():
    n_tr, n_te = CONFIG["subjects_per_cohort_train"], CONFIG["subjects_per_cohort_test"]
    train_subjects += subs[:n_tr]
    test_subjects += subs[n_tr:n_tr + n_te]

print(f"{sum(len(s) for s in cohorts.values())} subjects available, by cohort:")
for name, subs in cohorts.items():
    print(f"  {name:14s} {len(subs):3d}")
print(f"\ntraining on {len(train_subjects)} subjects, holding out {len(test_subjects)}")
assert not (set(train_subjects) & set(test_subjects)), "a subject is in both sets"
print("no subject is in both sets")

### 2.2 The classes, and their provenanceFive structures plus background. The provenance column is not decoration: it is thedifference between a label a person drew and a label a program produced. The numericcodes of the non-cortical structures follow FreeSurfer's colour table, as recorded in[`DATA.md`](DATA.md).

In [ ]:
CLASSES = [
    # index, name,                     provenance,              source codes
    (0, "background",                  "-",                     ()),
    (1, "cerebral cortex",             "manual (DKT protocol)", "labels.DKT31.manual >= 1000"),
    (2, "cerebral white matter",       "automatic (FreeSurfer aseg)", (2, 41)),
    (3, "lateral ventricle",           "automatic (FreeSurfer aseg)", (4, 43)),
    (4, "thalamus",                    "automatic (FreeSurfer aseg)", (10, 49)),
    (5, "hippocampus",                 "automatic (FreeSurfer aseg)", (17, 53)),
]
N_CLASSES = len(CLASSES)
MANUAL = {1}            # the only class a human drew

for i, name, prov, codes in CLASSES:
    print(f"{i}  {name:24s} {prov:30s} {codes}")

In [ ]:
import nibabel as nib

def build_labels(aseg, manual):
    """Map the dataset's label codes onto the six teaching classes."""
    y = np.zeros(aseg.shape, np.uint8)
    for idx, _, _, codes in CLASSES:
        if isinstance(codes, tuple) and codes:
            for c in codes:
                y[aseg == c] = idx
    y[manual >= 1000] = 1          # manual cortex wins where the two overlap
    return y


def load_subject(subject, n_slices, size):
    """Return (slices, labels) as arrays of shape (n, size, size)."""
    img = nib.as_closest_canonical(nib.load(subject / "t1weighted_brain.nii.gz"))
    aseg = np.asarray(nib.as_closest_canonical(
        nib.load(subject / "labels.DKT31.manual+aseg.nii.gz")).dataobj).astype(np.int32)
    manual = np.asarray(nib.as_closest_canonical(
        nib.load(subject / "labels.DKT31.manual.nii.gz")).dataobj).astype(np.int32)
    vol = img.get_fdata(dtype=np.float32)

    y = build_labels(aseg, manual)
    lo, hi = np.percentile(vol[vol > 0], (1, 99))       # per subject, so scanners differ less
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    keep = [k for k in range(y.shape[2]) if (y[:, :, k] > 0).sum() > 4000]
    keep = keep[:: max(1, len(keep) // n_slices)][:n_slices]

    half = size                                         # crop 2*size mm, then take every 2nd voxel
    cx, cy = vol.shape[0] // 2, vol.shape[1] // 2
    X, Y = [], []
    for k in keep:
        a = vol[max(0, cx - half):cx + half, max(0, cy - half):cy + half, k]
        b = y[max(0, cx - half):cx + half, max(0, cy - half):cy + half, k]
        pad = ((0, max(0, 2 * half - a.shape[0])), (0, max(0, 2 * half - a.shape[1])))
        X.append(np.pad(a, pad)[::2, ::2])
        Y.append(np.pad(b, pad)[::2, ::2])
    return np.stack(X).astype(np.float32), np.stack(Y).astype(np.int64)


def load_set(subjects):
    xs, ys = zip(*(load_subject(s, CONFIG["slices_per_subject"], CONFIG["image_size"])
                   for s in subjects))
    return np.concatenate(xs), np.concatenate(ys)

In [ ]:
import time

t0 = time.time()
X_train, Y_train = load_set(train_subjects)
print(f"{len(X_train)} training slices of {X_train.shape[1]}x{X_train.shape[2]}"
      f"  ({time.time() - t0:.1f}s)")

share = np.bincount(Y_train.reshape(-1), minlength=N_CLASSES) / Y_train.size
print("\nhow much of the image each class occupies:")
for (i, name, prov, _), s in zip(CLASSES, share):
    print(f"  {name:24s} {100 * s:5.2f}%   {prov}")

The imbalance is the whole difficulty of this task. The hippocampus is a few tenths ofa percent of the image; a model that simply never predicted it would still be rightalmost everywhere. Two things answer that below: a loss that weights the rare classes up,and a metric (Dice) that does not reward predicting nothing.

## 3. A small modelA small U-Net: three levels, eight channels at the top, about thirty thousandparameters. That is deliberate — the atlas ships one small model per organ, readable andretrainable on a laptop, rather than one large model for the whole body.

In [ ]:
import torch.nn as nn

class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class TinyUNet(nn.Module):
    def __init__(self, n_classes=N_CLASSES, base=CONFIG["base_channels"]):
        super().__init__()
        b = base
        self.enc1, self.enc2, self.enc3 = Block(1, b), Block(b, 2 * b), Block(2 * b, 4 * b)
        self.pool = nn.MaxPool2d(2)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), Block(4 * b, 2 * b)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * b, b, 2, 2), Block(2 * b, b)
        self.head = nn.Conv2d(b, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(e3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


model = TinyUNet()
print(f"{sum(p.numel() for p in model.parameters()):,d} parameters")

## 4. TrainingCross-entropy, weighted by the inverse square root of each class's share, so thehippocampus is not drowned by the background. The demonstrator runs a fixed number ofepochs on the sampled slices; nothing is tuned on the held-out subjects.

In [ ]:
weights = 1.0 / np.sqrt(share)
weights = torch.tensor(weights / weights.mean(), dtype=torch.float32)
print("loss weights:", " ".join(f"{name.split()[0]}={w:.2f}"
                                for (_, name, _, _), w in zip(CLASSES, weights)))

optimiser = torch.optim.Adam(model.parameters(), CONFIG["learning_rate"])
criterion = nn.CrossEntropyLoss(weight=weights)

Xt = torch.from_numpy(X_train)[:, None]
Yt = torch.from_numpy(Y_train)

t0 = time.time()
history = []
model.train()
for epoch in range(CONFIG["epochs"]):
    order = torch.randperm(len(Xt))
    total = 0.0
    for i in range(0, len(Xt), CONFIG["batch_size"]):
        idx = order[i:i + CONFIG["batch_size"]]
        optimiser.zero_grad()
        loss = criterion(model(Xt[idx]), Yt[idx])
        loss.backward()
        optimiser.step()
        total += loss.item() * len(idx)
    history.append(total / len(Xt))
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"  epoch {epoch + 1:3d}   loss {history[-1]:.4f}   ({time.time() - t0:.0f}s)")
print(f"trained in {time.time() - t0:.0f}s on CPU")

## 5. Evaluation: the model of recordThe numbers that count for this use do not come from the training above. They come from[`colab_train_v2.py`](colab_train_v2.py) on a GPU: a residual 3D U-Net, seventeenstructures, whole volumes, and — the point of version 2 — **measured against the labels aperson drew**, not only against FreeSurfer.It is trained in two stages. First on 81 brains with the automatic labels, none of themfrom the cohort that has expert labels. Then on the 20 that do, in four folds, so thateach of those twenty is scored by a model that never saw it. A fifth fine-tuning on alltwenty gives the single model delivered for this use; its performance is thatcross-validated estimate, not a measurement on brains held out from it.That run is recorded in [`results_v2.json`](results_v2.json), which this notebook readsrather than restates, so the notebook, the figure and the model card cannot disagree.

In [ ]:
import json

RESULTS = json.loads(Path("results_v2.json").read_text())
env, cfg_run, split = RESULTS["environment"], RESULTS["config"], RESULTS["split"]
s1, s3, counts = RESULTS["stage1"], RESULTS["stage3"], RESULTS["split"]["counts"]
m = RESULTS["model"]

print(f"{m['architecture']}: {m['parameters']:,d} parameters, patch {m['patch']}^3, "
      f"{m['outputs']} outputs")
print(f"  {env['device']}, torch {env['torch']}, seed {cfg_run['seed']}")
print(f"  stage 1  {counts['pretrain']:3d} brains, automatic labels   "
      f"{s1['steps']:5d} steps in {s1['seconds'] / 60:.0f} min, "
      f"best patch Dice {s1['best_patch_dice']}")
print(f"  stage 2  {counts['expert']:3d} brains, expert labels      "
      f"{len(split['expert_folds'])} folds of "
      f"{counts['per_fold']['train']}/{counts['per_fold']['val']}/"
      f"{counts['per_fold']['test']} train/val/test")
print(f"  stage 3  the delivered model            {s3['steps']:5d} steps on all "
      f"{len(s3['subjects'])}, selection: {s3['selection']}")

### What this record holds, and what it does notThe run wrote its results on a Colab runtime that was torn down before the file wascopied off it, so `results_v2.json` was rebuilt from the run's own printed summary. Thecell below prints what that means, from the file itself: what is authentic, what wasre-derived from the seed, and what is simply absent. Nothing missing was filled in byguessing — the per-subject spread is `null`, not reconstructed.

In [ ]:
prov = RESULTS["provenance_of_this_file"]
print(f"status: {prov['status']}\n")
for line in prov["authentic_here"]:
    print(f"  authentic  {line}")
print()
for line in prov["derived_not_read"]:
    print(f"  derived    {line.split(':')[0]}")
print()
for key, why in prov["absent_and_not_invented"].items():
    print(f"  absent     {key}")
print(f"\n{prov['do_not_do']}")

### Against the labels a person drewThe only scores here that may be called *correct*. Dice is the overlap; HD95 and ASSD aredistances **in millimetres**, read from the NIfTI affine, between the predicted surfaceand the expert's. Neither distance is ever optimised — the loss uses a boundary-weightedcross-entropy — so they stay an independent measure.

In [ ]:
man = RESULTS["results"]["against_manual"]
after, before = man["after_finetune"], man["before_finetune"]
order = sorted(after, key=lambda n: -after[n]["dice"]["mean"])

print(f"{'structure':24s} {'stage 1':>8s} {'tuned':>8s} {'gain':>7s} "
      f"{'HD95 mm':>8s} {'ASSD mm':>8s}")
for n in order:
    a, b = after[n]["dice"]["mean"], before[n]["dice"]["mean"]
    print(f"{n:24s} {b:8.3f} {a:8.3f} {a - b:+7.3f} "
          f"{after[n]['hd95_mm']['mean']:8.2f} {after[n]['assd_mm']['mean']:8.2f}")

bs = [before[n]["dice"]["mean"] for n in order]
as_ = [after[n]["dice"]["mean"] for n in order]
print(f"{'mean of the %d' % len(order):24s} {sum(bs) / len(bs):8.3f} "
      f"{sum(as_) / len(as_):8.3f} {(sum(as_) - sum(bs)) / len(as_):+7.3f}")
print(f"\nevery structure improved: "
      f"{all(after[n]['dice']['mean'] > before[n]['dice']['mean'] for n in order)}")
print(f"every average surface distance under one voxel: "
      f"{all(after[n]['assd_mm']['mean'] < 1.0 for n in order)}")

### Against FreeSurfer, kept separateThe same model, the same brains, scored against the automatic labels instead. This is**agreement, not correctness**, and it is never averaged with the table above. Read thearrow: where a structure moves down, the model left FreeSurfer as it moved toward theperson.

In [ ]:
aseg = RESULTS["results"]["against_aseg"]
rows = sorted(aseg["after_finetune"].items(), key=lambda kv: -(kv[1]["dice"]["mean"] or 0))
for n, row in rows:
    b, a = aseg["before_finetune"][n]["dice"]["mean"], row["dice"]["mean"]
    moved = b is not None and a < b
    shown = f"{b:6.3f}" if b is not None else "    --"
    print(f"  {n:24s} {shown} -> {a:6.3f}   {row['provenance']:15s}"
          f"{'  <- moved away' if moved else ''}")

# Count only what can honestly be counted: a structure needs an expert score AND a
# FreeSurfer score for "left one to follow the other" to mean anything. The vermis has
# no FreeSurfer counterpart; cerebral white matter has no expert one.
both = [n for n, row in rows
        if row["provenance"].startswith("manual")
        and aseg["before_finetune"][n]["dice"]["mean"] is not None]
away = [n for n in both
        if aseg["after_finetune"][n]["dice"]["mean"] < aseg["before_finetune"][n]["dice"]["mean"]]
print(f"\n{len(away)} of the {len(both)} structures scored against both references moved "
      f"away from FreeSurfer as they moved toward the expert.")
print(f"the other {len(both) - len(away)}: "
      f"{', '.join(n for n in both if n not in away)}.")
print("cerebral white matter is scored against FreeSurfer alone, and hardly moved: "
      "it is supervised from aseg in both stages.")
print(RESULTS["results"]["read_this_first"][1])

![Dice from the aseg-trained model to the expert-tuned one, and the boundary distance in millimetres, per structure](../../../docs/figures/png/brain/04_use1_v2_results.png)The figure puts both panels side by side: on the left the move from stage 1 to the tunedmodel against the expert, with a hollow mark for the same model against FreeSurfer; onthe right the boundary in millimetres, with the dashed line at one voxel.

### The demonstrator's own numbersFor completeness, what the small model trained above scores on its own held-outsubjects. It is a check that the pipeline works, not a result: fewer subjects, fewerslices, six structures, a fraction of the parameters, and two dimensions instead ofthree.

In [ ]:
def dice(pred, truth, cls):
    p, g = pred == cls, truth == cls
    denom = p.sum() + g.sum()
    return np.nan if denom == 0 else float(2 * (p & g).sum() / denom)


model.eval()
demo_rows, names = [], []
with torch.no_grad():
    for subject in test_subjects:
        x, y = load_subject(subject, CONFIG["slices_per_subject"], CONFIG["image_size"])
        pred = model(torch.from_numpy(x)[:, None]).argmax(1).numpy()
        demo_rows.append([dice(pred, y, c) for c in range(1, N_CLASSES)])
        names.append(subject.name)

demo_scores = np.array(demo_rows)
print(f"{'demonstrator':16s} | " + " | ".join(f"{n.split()[-1][:9]:>9s}" for _, n, _, _ in CLASSES[1:]))
for name, row in zip(names, demo_scores):
    print(f"{name:16s} | " + " | ".join(f"{v:9.3f}" for v in row))
print(f"{'mean':16s} | " + " | ".join(f"{v:9.3f}" for v in np.nanmean(demo_scores, 0)))

## 6. The errors, looked at**The millimetres say what Dice hides.** Dice punishes a small structure far harder thana large one for the same mistake: erode the truth by a single voxel and the pallidumalready falls to about 0.74, while its surface has moved exactly one voxel. So read thetwo columns together. In the table above, the lowest Dice values belong to the smalleststructures — the third ventricle is 0.024 % of the voxels in a brain, the accumbens0.032 % — and yet **every average surface distance is under one voxel**. The boundary isnearly right; there is simply very little of it to be right about.**The cerebellar vermis is the clearest thing the run shows.** FreeSurfer has no vermislabel at all: it folds that tissue into the cerebellar cortex. So stage 1 never saw one,and scored 0.000. After fine-tuning on twelve brains it reaches 0.804, with its boundary0.90 mm out on average — and it is also the worst structure by HD95 at 2.57 mm, which iswhat learning against what stage 1 taught looks like. A structure a program cannot isolate,learned from a dozen examples a person labelled: that is the argument for expert labels inone line.**Leaving FreeSurfer is the cost, and it is not a regression.** Thirteen structures agreeless with aseg after fine-tuning than before. Where a program and a person draw a boundarydifferently, a model cannot follow both. This one was asked to follow the person, and theaseg column is kept so that the trade is visible rather than hidden.The cells below show, on the notebook's own demonstrator, where a segmentation model'sdisagreement actually lives: on the boundaries.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

worst_demo = int(np.nanmean(demo_scores, 1).argmin())
subject = test_subjects[worst_demo]
x, y = load_subject(subject, CONFIG["slices_per_subject"], CONFIG["image_size"])
with torch.no_grad():
    pred = model(torch.from_numpy(x)[:, None]).argmax(1).numpy()

k = int(np.argmax([(y[i] > 0).sum() for i in range(len(y))]))
cmap = ListedColormap(["#0d0d0c", "#3987e5", "#6f6e66", "#c3c2b7", "#d95926", "#f8f7f4"])

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(x[k].T, cmap="gray", origin="lower"); axes[0].set_title("T1 slice")
axes[1].imshow(y[k].T, cmap=cmap, vmin=0, vmax=5, origin="lower")
axes[1].set_title("dataset labels\n(cortex manual, rest FreeSurfer)")
axes[2].imshow(pred[k].T, cmap=cmap, vmin=0, vmax=5, origin="lower")
axes[2].set_title("demonstrator output")
for a in axes:
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"the notebook's demonstrator on {subject.name} - not the model of record")
plt.tight_layout()
plt.show()

In [ ]:
# where the disagreement sits: boundary voxels against interior voxels
def interior_of(mask):
    """Erode a 2D mask by one voxel, with numpy alone (4-neighbour)."""
    up, down = np.roll(mask, 1, 0), np.roll(mask, -1, 0)
    left, right = np.roll(mask, 1, 1), np.roll(mask, -1, 1)
    inner = mask & up & down & left & right
    inner[0, :] = inner[-1, :] = inner[:, 0] = inner[:, -1] = False
    return inner


boundary_hits = interior_hits = boundary_total = interior_total = 0
for cls in range(1, N_CLASSES):
    g = y == cls
    interior = np.stack([interior_of(g[i]) for i in range(len(g))])
    boundary = g & ~interior
    boundary_hits += int((pred[boundary] == cls).sum()); boundary_total += int(boundary.sum())
    interior_hits += int((pred[interior] == cls).sum()); interior_total += int(interior.sum())

print("on this subject, the demonstrator labels correctly:")
print(f"  {100 * interior_hits / max(interior_total, 1):5.1f}% of interior voxels")
print(f"  {100 * boundary_hits / max(boundary_total, 1):5.1f}% of boundary voxels")
print("\nthe disagreement lives on the boundaries, which is why the model of record is")
print("measured in millimetres there, and why the loss weights those voxels more")

## 7. What this is, and what it is not**Two models live in this folder, and they are not the same thing.**| | This notebook | The model of record ||---|---|---|| purpose | read the pipeline end to end in a minute | the numbers for this use || geometry | 2.5D slices, 96x96 | 3D patches, 96x96x96, whole volumes || subjects | 15, capped by `CONFIG` | 81 pre-train, then 20 with expert labels || structures | 6 | 17 || subcortical truth | FreeSurfer only | labels a person drew, for the 20 || model | about 30,000 parameters | 3,225,966 parameters || hardware | a laptop CPU, about a minute | a Tesla T4, about two hours || recorded in | nothing: it is a demonstration | `results_v2.json`, `MODEL_CARD.md` |To reproduce the model of record: `python colab_train_v2.py` on a GPU runtime. The seedand the configuration are in the script, it fetches both records itself and checks everychecksum before it starts. `--inventory` lists every label code in the data and whatbecomes of it; `--smoke` runs the whole pipeline small enough for a CPU.**How to quote it.** The twenty expert scores come from four models, one per fold, eachscoring the five brains it never saw. The delivered model is fine-tuned on all twenty, sonothing is held out from it: its performance is that cross-validated estimate, not ameasurement on unseen brains.**What neither model is.** Not a validated segmentation tool, not a statement about anyindividual, and not something to decide anything about a person with. They describehealthy adult brains from one dataset. Where a score is measured against FreeSurfer itmeans agreement, nothing more — and cerebral white matter can only ever be scored thatway here.**The weights.** Not published, and version 2 makes that stricter rather than looser: theT1 images carry the terms of OASIS, NKI and MMRR, and the expert subcortical labels areCC BY-NC-ND 4.0 — no commercial use, no distribution of modified versions. They areregenerable from the script and the seed. See [`DATA.md`](DATA.md), the[model card](MODEL_CARD.md) and the[licence rule](../../../../CATALOGUE.md#two-licences-kept-apart).**Notebook outputs are not committed.** Rendering a slice puts a piece of the MRI in thefile. The data stays with its provider: run the notebook to see the images.